# Project 1 — Vocabulary and architecture experiments

A separate local copy of [project1.ipynb](project1.ipynb), preserving its baseline and results. Run top to bottom, like [Tutorial 1](tutorial.ipynb). All repeated functions still come from `utils/p1/functions.py` and the course modules.

**Question:** does a larger BPE vocabulary help, and does changing MLP width or regularization improve generalization?

We keep the same 3,000 examples and five folds. Compare the original nine systems at **1,500, 4,000 and 8,000 tokens**, then isolate three MLP changes at the fixed middle size (4,000). This is a bounded exploratory study, not a guarantee of improved accuracy. The old folds have already informed our hypotheses, so this is not an independent confirmation. The official test is never loaded.

The original notebook remains the required 3 × 3 baseline. This copy is its extension; nothing is published to GitHub.

## 1. Setup

Like the tutorial, we first make the repository importable and load the libraries. Select the project's `.venv` kernel. We use CPU execution and a fixed seed so the run can be repeated.

In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "part1/tutorial.ipynb").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Keep generated caches in this local, Git-ignored directory.
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(root / ".cache"))
os.environ.setdefault("HF_HUB_CACHE", str(root / ".cache/huggingface/hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:
import hashlib
import json
import random
import time
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

import part1
from utils.p1.functions import (
    TokenIndicators, BagOfWords, TFIDF, encode_texts,
    TwoHiddenMLP, train_mlp, predict_mlp, classification_scores,
)

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

### Settings and comparison plan

The demonstrations use the original 1,500-token settings. The full experiment below compares `[1500, 4000, 8000]`, holding everything else fixed. Three additional **count-feature MLPs at 4,000 tokens** change just one setting each: width 64 → 32; dropout 0.2 → 0.5; or weight decay 0.0001 → 0.001.

We specify these choices before running the new comparisons. We report every configuration, including any that get worse. No stopping epoch, threshold or configuration is chosen from an outer fold during training. Ranking many configurations afterwards is still exploratory model selection; freeze a choice and use a fresh evaluation before claiming its final performance.

In [ ]:
use_toy_data = False
seed = 42
num_folds = 5
examples_per_class = 1500
vocab_size = 1500
min_frequency = 2
num_epochs = 10
hidden_dim = 64
second_hidden_dim = 32
dropout = 0.2
batch_size = 64
learning_rate = 0.001
weight_decay = 0.0001
rope = 0.01

dataset_revision = "d29c47f41c8b51348b5c8e8c81c039b3132b66d1"
if use_toy_data:
    examples_per_class, vocab_size, num_epochs = 20, 128, 2

run_label = "SMOKE_NOT_SCIENTIFIC" if use_toy_data else "PROJECT1_EXPLORATORY"
results_folder = root / "part1/results" / ("toy_experiments" if use_toy_data else "project1_experiments")
results_folder.mkdir(parents=True, exist_ok=True)
print(run_label)

vocabulary_sizes = [128, 256, 512] if use_toy_data else [1500, 4000, 8000]
architecture_vocab = vocabulary_sizes[1]

## 2. Load the data

We predict `prompt_harm_label` from `prompt` only. The subcategory is used to filter rows, never as a model feature. We do not load response text into our working table.

WildGuardMix requires accepting its terms and a Hugging Face login. The setup notebook explains this. **Only `wildguardtrain/train` is loaded. The official test stays untouched.** We pin the dataset revision and also verify it if Hugging Face uses a local cache.

We display shapes, counts and safe teaching examples throughout this notebook, rather than raw dataset prompts.

In [ ]:
if use_toy_data:
    records = []
    for i in range(examples_per_class):
        records.append({"prompt": f"Garden notebook {i}: water the flowers and prepare fresh soil.",
                        "prompt_harm_label": "unharmful", "subcategory": "synthetic"})
        records.append({"prompt": f"Space notebook {i}: observe the planets through a telescope tonight.",
                        "prompt_harm_label": "harmful", "subcategory": "synthetic"})
    data = pd.DataFrame(records)
else:
    dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtrain", split="train",
        revision=dataset_revision, token=True,
        cache_dir=str(root / ".cache/huggingface/datasets"),
    )
    assert dataset.cache_files and all(
        dataset_revision in Path(item["filename"]).parts for item in dataset.cache_files
    ), "The cached dataset does not match our pinned revision."
    data = dataset.select_columns(["prompt", "prompt_harm_label", "subcategory"]).to_pandas()

print("Rows loaded:", len(data))

### 2.1 Apply the agreed category filter first

Remove an exact `strip().casefold()` match to `toxic_language_hate_speech`. Then remove missing harmfulness labels and map `unharmful` to 0 and `harmful` to 1.

The team audit expects **86,759 raw rows → 83,739 eligible rows**, before prompt cleanup. We check those counts instead of silently accepting a different dataset.

In [ ]:
audit = {"status": run_label, "original_rows": len(data)}
excluded = data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech")
audit["excluded_category_rows"] = int(excluded.sum())
data = data.loc[~excluded].copy()
assert not data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech").any()

labels = data["prompt_harm_label"].str.strip().str.casefold()
missing_label = labels.isna() | labels.eq("")
audit["missing_label_rows"] = int(missing_label.sum())
assert labels.loc[~missing_label].isin(["unharmful", "harmful"]).all(), "Unexpected label."
data = data.loc[~missing_label].copy()
data["label"] = labels.loc[~missing_label].map({"unharmful": 0, "harmful": 1}).astype(int)
audit["eligible_rows"] = len(data)
audit["eligible_unharmful"] = int(data["label"].eq(0).sum())
audit["eligible_harmful"] = int(data["label"].eq(1).sum())

if not use_toy_data:
    expected = {"original_rows": 86759, "excluded_category_rows": 3020,
                "missing_label_rows": 0, "eligible_rows": 83739,
                "eligible_unharmful": 40543, "eligible_harmful": 43196}
    assert all(audit[key] == value for key, value in expected.items()), "Dataset audit mismatch."
display(pd.Series(audit, name="Data audit"))

### 2.2 Clean and deduplicate prompts

Normalize Unicode to NFC and strip edge whitespace. Preserve case and internal whitespace. Remove empty text. If the same normalized prompt has conflicting labels, drop the whole group; otherwise keep one copy.

This prevents exact copies of a prompt from appearing in both training and validation. Paraphrases and related scenarios can still cross folds, which remains a limitation. Removal counts below are separate and add up.

In [ ]:
data["prompt"] = data["prompt"].fillna("").str.normalize("NFC").str.strip()
empty_prompt = data["prompt"].eq("")
audit["empty_prompt_rows"] = int(empty_prompt.sum())
data = data.loc[~empty_prompt].copy()

conflicting = data.groupby("prompt")["label"].transform("nunique").gt(1)
audit["contradictory_label_rows"] = int(conflicting.sum())
audit["contradictory_label_groups"] = int(data.loc[conflicting, "prompt"].nunique())
data = data.loc[~conflicting].copy()
audit["duplicate_rows"] = int(data.duplicated("prompt").sum())
data = data.drop_duplicates("prompt").copy()
audit["retained_rows"] = len(data)

removals = ["excluded_category_rows", "missing_label_rows", "empty_prompt_rows",
            "contradictory_label_rows", "duplicate_rows", "retained_rows"]
assert sum(audit[key] for key in removals) == audit["original_rows"]
assert data["prompt"].is_unique

data["id"] = data["prompt"].map(lambda text: "wg-" + hashlib.sha256(text.encode("utf-8")).hexdigest())
data = data[["id", "prompt", "label"]].sort_values("id").reset_index(drop=True)
display(pd.Series(audit, name="Completed audit"))

### 2.3 Take a reproducible sample

A balanced 3,000-example sample keeps the experiment practical on CPU. Its 50% harmful prevalence differs from the original training population (about 51.58% before cleanup). We report both.

The hash IDs let us save and match predictions without exporting prompt text. Hashing is identification, not encryption.

In [ ]:
rng = random.Random(seed)
chosen_indices = []
for label in [0, 1]:
    candidates = data.index[data["label"].eq(label)].tolist()
    assert len(candidates) >= examples_per_class
    chosen_indices.extend(rng.sample(candidates, examples_per_class))
rng.shuffle(chosen_indices)
sample = data.loc[chosen_indices].reset_index(drop=True)

texts = sample["prompt"].tolist()
y = sample["label"].to_numpy()
ids = sample["id"].to_numpy()
audit["sample_rows"] = len(sample)
audit["sample_harmful_prevalence"] = float(y.mean())
print("Sample class counts:", sample["label"].value_counts().sort_index().to_dict())
print("Eligible population harmful prevalence:", audit["eligible_harmful"] / audit["eligible_rows"])
display(sample["prompt"].str.len().describe(percentiles=[0.5, 0.95]).rename("Prompt length in characters"))

## 3. Create five shared folds

We use `StratifiedKFold` to keep both labels represented in every fold. The tutorial's splitting helper shuffles but is not stratified. Every system will reuse this **same partition**.

We create the folds before learning BPE or IDF. Training those on the entire sample would let validation text influence the features.

In [ ]:
splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
folds = list(splitter.split(texts, y))
fold_assignment = np.full(len(sample), -1)
for fold, (train_indices, validation_indices) in enumerate(folds):
    assert set(train_indices).isdisjoint(validation_indices)
    fold_assignment[validation_indices] = fold
    print(f"Fold {fold + 1}: {len(train_indices)} train, {len(validation_indices)} validation")
assert (fold_assignment >= 0).all()

# Start with one fold to understand each component, like the tutorial.
train_indices, validation_indices = folds[0]
train_texts = [texts[i] for i in train_indices]
validation_texts = [texts[i] for i in validation_indices]
y_train, y_validation = y[train_indices], y[validation_indices]

# Same rows, labels and folds as the original run: no easier resampling.
baseline_folder = root / "part1/results" / ("toy" if use_toy_data else "project1")
if (baseline_folder / "folds.csv").exists():
    saved_folds = pd.read_csv(baseline_folder / "folds.csv")
    pd.testing.assert_frame_equal(saved_folds,
        pd.DataFrame({"id": ids, "gold": y, "fold": fold_assignment}))
    print("Verified: exactly the baseline sample and fold assignments.")


## 4. Tokenization: one BPE algorithm, three vocabulary sizes

A tokenizer splits text into units. **BPE (byte-pair encoding)** starts from characters in this course implementation and repeatedly merges frequent adjacent units. For example, character pieces can gradually become `plant` and `ing`; the actual merges depend on the training corpus. Metaspace marks word boundaries with `▁`.

- **1,500 tokens:** fewer available units; words often need more pieces. Features share evidence across words, but small fragments may be ambiguous.
- **4,000 tokens:** allows more longer pieces, with more distinct input features.
- **8,000 tokens:** can preserve more whole words, but rare features get fewer examples, and an MLP needs more first-layer weights.

`vocab_size` is a training target, not the number of words a model can read or a text-length limit. Learned size can differ because of the initial alphabet and frequency constraint. We measure actual size. A new word can often use known subword pieces; an unseen character can still become `[UNK]` here. This is character-based BPE, not guaranteed byte-fallback tokenization.

We directly reuse `part1.experiment.train_tokenizer`, fitted only on each training fold. Below is a safe teaching sentence.

In [ ]:
tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
safe_sentence = "Please explain how plants grow in a garden."
print(tokenizer.encode(safe_sentence).tokens)
print("Learned vocabulary size:", tokenizer.get_vocab_size())

## 5. The three representations — not three tokenizers

The pipeline is **text → BPE tokens → numeric vector → classifier**. All three representations share exactly the same BPE tokenizer within a fold.

For an illustrative token sequence `[garden, garden, plants]`, with vocabulary `[garden, plants, moon]`:

| Representation | Vector or rule | Meaning | Main limitation |
|---|---|---|---|
| Binary indicators | `[1, 1, 0]` | Which tokens appeared? | Repetition is discarded. |
| Bag-of-words counts | `[2, 1, 0]` | How often did each appear? | Length/repetition can dominate. |
| TF-IDF | counts × training-corpus IDF, then L2 normalization | Give common tokens less relative weight | Rare does not automatically mean useful. |

Our IDF is `log((1 + n_train) / (1 + df_train)) + 1`. A token in every training document has IDF 1, not 0. The table below shows real BPE features for a safe example.

**All three lose order.** A deeper MLP cannot reconstruct information that its input vector has discarded. For example, word-count vectors for “cats chase dogs” and “dogs chase cats” are identical. Local token n-grams would add some order information, but are a separate representation extension, not part of this controlled comparison.

If you meant three *families of tokenizers*: character tokenization splits into characters, word tokenization into word-like units, and subword BPE into learned pieces. We currently implement BPE only; indicators/counts/TF-IDF are the three required representations.

In [ ]:
indicator_encoder = TokenIndicators(tokenizer)
bow_encoder = BagOfWords(tokenizer)
tfidf_encoder = TFIDF(tokenizer).fit(train_texts)
encoders = {"indicators": indicator_encoder, "bow": bow_encoder, "tfidf": tfidf_encoder}

safe_repetition = "garden garden garden plants"
token_ids = np.flatnonzero(bow_encoder(safe_repetition))
display(pd.DataFrame({
    "token": [tokenizer.id_to_token(int(i)) for i in token_ids],
    "indicator": indicator_encoder(safe_repetition)[token_ids],
    "count": bow_encoder(safe_repetition)[token_ids],
    "tfidf": tfidf_encoder(safe_repetition)[token_ids],
}))
assert bow_encoder(safe_repetition).max() > 1
assert set(indicator_encoder(safe_repetition)).issubset({0, 1})

### Represent the first fold

For the next three examples, hold the representation fixed at TF-IDF. `encode_texts` applies an encoder to each text and creates the SciPy sparse matrix expected by the course models.

Transforming validation text must not update the tokenizer or IDF. The assertions check that explicitly.

In [ ]:
vocabulary_before = tokenizer.get_vocab().copy()
idf_before = tfidf_encoder.transformer.idf_.copy()
X_train = encode_texts(train_texts, tfidf_encoder)
X_validation = encode_texts(validation_texts, tfidf_encoder)
assert tokenizer.get_vocab() == vocabulary_before
np.testing.assert_array_equal(tfidf_encoder.transformer.idf_, idf_before)
print("Feature matrix shapes:", X_train.shape, X_validation.shape)

## 6. Log-linear classifier

Like `part1.loglin`, we use sklearn logistic regression. Here we pass the feature matrices directly because we have already encoded the texts. This avoids repeating the tutorial wrapper's encoding step. `lbfgs` fits the same regularized logistic model family and works well with these small sparse matrices.

The two probability columns correspond to `[unharmful, harmful]`. The decision rule is the tutorial's most-probable-class rule. An exact tie goes to class 0.

In [ ]:
logistic_model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=seed)
logistic_model.fit(X_train, y_train)
logistic_probabilities = logistic_model.predict_proba(X_validation)
logistic_predictions = logistic_probabilities.argmax(axis=1)
first_fold_scores = {"logistic": classification_scores(y_validation, logistic_predictions)}
display(pd.Series(first_fold_scores["logistic"]))

## 7. The tutorial's single-hidden-layer MLP

We instantiate `part1.smlp.SparseMLP` directly: sparse input, 64 ReLU units, dropout and log-softmax output. The reusable training helper calls the course `SparseDataset`, `sparse_collate` and `train_one_epoch`.

We train for a fixed number of epochs. The held-out fold does not choose a checkpoint or stopping epoch. Negative log-likelihood consumes the model's log probabilities.

In [ ]:
torch.manual_seed(seed)
mlp_model = part1.smlp.SparseMLP(X_train.shape[1], hidden_dim, num_classes=2, dropout=dropout)
training_history = train_mlp(
    mlp_model, X_train, y_train, epochs=num_epochs, seed=seed,
    batch_size=batch_size, learning_rate=learning_rate, weight_decay=weight_decay,
)
mlp_probabilities = predict_mlp(mlp_model, X_validation)
first_fold_scores["mlp1"] = classification_scores(y_validation, mlp_probabilities.argmax(axis=1))
display(pd.DataFrame(training_history).rename_axis("epoch"))

## 8. Our extra architecture: two hidden layers

`TwoHiddenMLP` extends the official `SparseMLP`. It keeps its sparse first layer and changes the output module to a second 32-unit ReLU layer followed by the class output.

We keep dropout, optimizer and training budget the same. This changes both architecture and parameter count; it is not a parameter-matched comparison.

In [ ]:
torch.manual_seed(seed)
modified_mlp = TwoHiddenMLP(X_train.shape[1], hidden_dim, second_hidden_dim, dropout)
modified_history = train_mlp(
    modified_mlp, X_train, y_train, epochs=num_epochs, seed=seed,
    batch_size=batch_size, learning_rate=learning_rate, weight_decay=weight_decay,
)
modified_probabilities = predict_mlp(modified_mlp, X_validation)
first_fold_scores["mlp2"] = classification_scores(y_validation, modified_probabilities.argmax(axis=1))
display(pd.DataFrame(first_fold_scores).T)
print("These are first-fold demonstrations. The complete comparison comes next.")

## 9. Why these architecture changes?

The original models remain logistic regression, a 64-unit MLP, and a 64 → 32 MLP. With vocabulary size V, the one-layer MLP has **V × 64 + 194** trainable parameters: 96,194 at V=1,500 versus 512,194 at V=8,000. Sparse inputs do not make its weight matrix sparse.

The existing results do not establish that greater depth helps. We therefore test three simple alternatives at V=4,000 using counts, the original descriptive leading representation:

1. **32 hidden units:** reduce capacity; hypothesis: less memorization. Only width changes.
2. **Dropout 0.5:** discourage dependence on individual hidden units. Only dropout changes.
3. **Weight decay 0.001:** penalize large weights more. Only weight decay changes.

Dropout and weight decay are regularization changes, not new architectures. All use the same 10 epochs, optimizer, seed and folds. We also measure final training F1 with dropout disabled, making the train–validation gap interpretable. A smaller gap alone is not success: a model may simply underfit.

This gives **27 original configurations + 3 additional controls = 30 configurations × 5 folds = 150 fits** (plus the demonstrations above).

In [ ]:
score_rows, prediction_rows, token_rows = [], [], []
model_names = ["logistic", "mlp1", "mlp2"]
representation_names = ["indicators", "bow", "tfidf"]

for fold, (train_indices, validation_indices) in enumerate(folds):
    fold_seed = seed + fold
    train_texts = [texts[i] for i in train_indices]
    validation_texts = [texts[i] for i in validation_indices]
    y_train, y_validation = y[train_indices], y[validation_indices]

    for requested_vocab in vocabulary_sizes:
        tokenizer = part1.experiment.train_tokenizer(train_texts, requested_vocab, min_frequency)
        actual_vocab = tokenizer.get_vocab_size()
        # Only lengths and counts are exported, never raw prompt tokens.
        lengths, unknown, token_total = [], 0, 0
        for text in validation_texts:
            token_ids = tokenizer.encode(text).ids
            lengths.append(len(token_ids))
            unknown += token_ids.count(tokenizer.token_to_id("[UNK]"))
            token_total += len(token_ids)
        token_rows.append({"fold": fold, "vocab_size": requested_vocab,
            "actual_vocab": actual_vocab, "mean_validation_tokens": np.mean(lengths),
            "unknown_token_rate": unknown / max(token_total, 1)})
        if fold == 0:
            print("Safe example, vocabulary", requested_vocab,
                  tokenizer.encode("Please explain how plants grow in a garden.").tokens)

        encoders = {"indicators": TokenIndicators(tokenizer),
                    "bow": BagOfWords(tokenizer),
                    "tfidf": TFIDF(tokenizer).fit(train_texts)}
        for representation, encoder in encoders.items():
            vocabulary_before = tokenizer.get_vocab().copy()
            X_train = encode_texts(train_texts, encoder)
            X_validation = encode_texts(validation_texts, encoder)
            assert tokenizer.get_vocab() == vocabulary_before
            variants = model_names.copy()
            if requested_vocab == architecture_vocab and representation == "bow":
                variants += ["mlp1_width32", "mlp1_dropout50", "mlp1_decay001"]

            for model_name in variants:
                started = time.perf_counter()
                width = 32 if model_name == "mlp1_width32" else hidden_dim
                model_dropout = 0.5 if model_name == "mlp1_dropout50" else dropout
                decay = 0.001 if model_name == "mlp1_decay001" else weight_decay
                if model_name == "logistic":
                    model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000,
                                               random_state=fold_seed)
                    model.fit(X_train, y_train)
                    fit_seconds = time.perf_counter() - started
                    probabilities = model.predict_proba(X_validation)
                    train_predictions = model.predict_proba(X_train).argmax(axis=1)
                    parameters = model.coef_.size + model.intercept_.size
                else:
                    torch.manual_seed(fold_seed)
                    if model_name == "mlp2":
                        model = TwoHiddenMLP(actual_vocab, hidden_dim, second_hidden_dim, dropout)
                    else:
                        model = part1.smlp.SparseMLP(actual_vocab, width, 2, model_dropout)
                    train_mlp(model, X_train, y_train, epochs=num_epochs, seed=fold_seed,
                              batch_size=batch_size, learning_rate=learning_rate, weight_decay=decay)
                    fit_seconds = time.perf_counter() - started
                    probabilities = predict_mlp(model, X_validation)
                    train_predictions = predict_mlp(model, X_train).argmax(axis=1)
                    parameters = sum(p.numel() for p in model.parameters())
                assert np.isfinite(probabilities).all()
                np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
                predictions = probabilities.argmax(axis=1)
                metrics = classification_scores(y_validation, predictions)
                training_f1 = classification_scores(y_train, train_predictions)["macro_f1"]
                score_rows.append({"model": model_name, "representation": representation,
                    "vocab_size": requested_vocab, "actual_vocab": actual_vocab, "fold": fold,
                    **metrics, "training_macro_f1": training_f1,
                    "generalization_gap": training_f1 - metrics["macro_f1"],
                    "parameters": parameters, "fit_seconds": fit_seconds, "seed": fold_seed})
                for j, index in enumerate(validation_indices):
                    prediction_rows.append({"id": ids[index], "fold": fold,
                        "model": model_name, "representation": representation,
                        "vocab_size": requested_vocab, "gold": int(y[index]),
                        "prediction": int(predictions[j]),
                        "probability_harmful": float(probabilities[j, 1])})
        print(f"Finished fold {fold + 1}/{num_folds}, vocabulary {requested_vocab}", flush=True)

scores = pd.DataFrame(score_rows)
oof = pd.DataFrame(prediction_rows)
token_diagnostics = pd.DataFrame(token_rows)
keys = ["model", "representation", "vocab_size"]
assert len(scores) == 150
assert scores.groupby(keys).size().eq(num_folds).all()
assert not oof.duplicated(["id", *keys]).any()
assert len(oof) == len(sample) * 30
assert oof.groupby("id")["fold"].nunique().eq(1).all()
print("Completed all 150 fits.")

# Checkpoint numeric outputs before rendering the report.
scores.to_csv(results_folder / "cv_scores.csv", index=False)
oof.to_csv(results_folder / "oof_predictions.csv", index=False)
token_diagnostics.to_csv(results_folder / "token_diagnostics.csv", index=False)

## 10. Read all results, including unsuccessful changes

Macro-F1 remains primary. Accuracy is easy to interpret on our balanced sample, while harmful recall and benign false-positive rate show which mistakes change. Fold standard deviation is descriptive, not an independent-sample confidence interval.

The tables compare vocabulary sizes within each model/representation, then the single-setting MLP controls at 4,000. A best-looking cell is a candidate for later validation, not a confirmed winning model.

The unknown-token rate uses the number of encoded tokens as its denominator. Larger vocabularies shorten sequences, so this rate can rise even without more unknown characters. It is not by itself evidence of worse character coverage.

In [ ]:
summary = scores.groupby(keys).agg(
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    accuracy=("accuracy", "mean"), harmful_recall=("harmful_recall", "mean"),
    benign_fpr=("benign_fpr", "mean"), training_macro_f1=("training_macro_f1", "mean"),
    generalization_gap=("generalization_gap", "mean"), parameters=("parameters", "mean"),
    fit_seconds=("fit_seconds", "mean"),
).reset_index()
display(summary[summary.model.isin(model_names)].pivot(
    index=["model", "representation"], columns="vocab_size", values="macro_f1_mean").round(4))
display(summary[(summary.vocab_size == architecture_vocab) &
                (summary.representation == "bow")].round(4))
display(token_diagnostics.groupby("vocab_size").mean(numeric_only=True).drop(columns="fold").round(5))

# A meaningful integration check: unchanged settings should reproduce saved predictions.
baseline_predictions_path = baseline_folder / "p1_oof_predictions.csv"
if baseline_predictions_path.exists():
    old = pd.read_csv(baseline_predictions_path)
    current = oof[(oof.vocab_size == vocabulary_sizes[0]) & oof.model.isin(model_names)]
    paired = old.merge(current, on=["id", "fold", "model", "representation"],
                       suffixes=("_old", "_new"), validate="one_to_one")
    assert len(paired) == len(sample) * 9
    assert (paired.prediction_old == paired.prediction_new).all()
    np.testing.assert_allclose(paired.probability_harmful_old,
                               paired.probability_harmful_new, atol=1e-6)
    print("Baseline reproduced: all original out-of-fold predictions match.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, representation in zip(axes, representation_names):
    for model_name in model_names:
        rows = summary[(summary.model == model_name) &
                       (summary.representation == representation)].sort_values("vocab_size")
        ax.plot(rows.vocab_size, rows.macro_f1_mean, marker="o", label=model_name)
    ax.set_title(representation)
    ax.set_xlabel("Requested BPE vocabulary size")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Mean validation macro-F1")
axes[-1].legend()
fig.tight_layout()
fig.savefig(results_folder / "vocabulary_comparison.png", dpi=150)
plt.show()

## 11. Paired differences and measured conclusions

Each changed system is compared with the matching original architecture/representation at 1,500 tokens. MLP controls are additionally compared with the original 64-unit MLP at 4,000 tokens. These paired differences isolate the intended setting changes; they do not correct the optimism of selecting a winner from many experiments.

We do not reuse the old two preselected Bayesian comparisons to claim certainty about these newly explored hypotheses. Formal confirmation needs a frozen configuration and a fresh evaluation (or an inner selection loop for future tuning). See [scikit-learn's explanation of nested evaluation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).

In [ ]:
reference = scores[(scores.vocab_size == vocabulary_sizes[0]) & scores.model.isin(model_names)]
paired_vocab = scores[scores.model.isin(model_names)].merge(
    reference[["model", "representation", "fold", "macro_f1"]],
    on=["model", "representation", "fold"], suffixes=("", "_baseline"), validate="many_to_one")
paired_vocab["delta_macro_f1"] = paired_vocab.macro_f1 - paired_vocab.macro_f1_baseline
vocabulary_deltas = paired_vocab.groupby(keys).delta_macro_f1.agg(["mean", "std"]).reset_index()
display(vocabulary_deltas[vocabulary_deltas.vocab_size != vocabulary_sizes[0]].round(4))

reference_mlp = scores[(scores.vocab_size == architecture_vocab) &
                      (scores.representation == "bow") & (scores.model == "mlp1")]
controls = scores[scores.model.str.startswith("mlp1_")].merge(
    reference_mlp[["fold", "macro_f1"]], on="fold", suffixes=("", "_reference"), validate="many_to_one")
controls["delta_macro_f1"] = controls.macro_f1 - controls.macro_f1_reference
architecture_deltas = controls.groupby("model").delta_macro_f1.agg(["mean", "std"]).reset_index()
display(architecture_deltas.round(4))

from IPython.display import Markdown
best = summary.loc[summary.macro_f1_mean.idxmax()]
base_summary = summary[(summary.vocab_size == vocabulary_sizes[0]) & summary.model.isin(model_names)]
base_best = base_summary.loc[base_summary.macro_f1_mean.idxmax()]
display(Markdown(
    f"**Exploratory leader:** {best['model']} / {best['representation']} / "
    f"{int(best['vocab_size']):,} tokens: macro-F1 **{best['macro_f1_mean']:.4f}**, "
    f"accuracy **{best['accuracy']:.2%}**. Original descriptive best F1: "
    f"{base_best['macro_f1_mean']:.4f}; observed difference "
    f"{best['macro_f1_mean'] - base_best['macro_f1_mean']:+.4f}.\n\n"
    f"Harmful recall: {best['harmful_recall']:.2%}; benign false-positive rate: "
    f"{best['benign_fpr']:.2%}. Final training F1: {best['training_macro_f1']:.4f}. "
    "This is a measured CV ranking, not an untouched-test result or a guaranteed gain."
))

## 12. Save locally and decide the next experiment

Outputs contain aggregate metrics and hashed IDs, not raw prompts or vocabulary dumps. The original notebook/results are not overwritten.

If larger vocabularies help, justify them by the measured paired changes and cost. If a smaller or more regularized MLP fails, report that: reduced capacity is a hypothesis, not automatically an improvement. Extra depth cannot recover word order lost by all three input representations.

Next useful work would be a training-size curve using more of the cleaned data, holding evaluation rows fixed; or a separately labelled n-gram representation experiment. For model selection, use an inner training/validation split and refit tokenizer/IDF there too. Do not repeatedly inspect the official test. Near-duplicate scenarios, imperfect labels, distribution shift and insufficient context can still limit results.

**Attribution:** reused course BPE, SparseMLP, sparse training and the original project helpers. OpenAI Codex assisted with this notebook; the team should review and understand every step.

**What this completed run supports:** increasing the vocabulary from 1,500 to 8,000 improved mean validation macro-F1 for each of the nine original systems. The one-layer MLP with indicators and the one-layer MLP with TF-IDF are essentially tied at 0.798 macro-F1 and 79.8% accuracy. At 4,000 tokens, the three single-setting controls improved count-MLP F1 by only about 0.0035–0.0049; those small observed gains do not establish a clear winner. Training F1 near 0.998 for the leading models still leaves about a 0.20 train–validation gap. Prefer a larger-vocabulary one-layer MLP as a candidate for further validation; the results do not justify adding depth just to obtain a better score.

Reference explanations: [BPE training parameters](https://huggingface.co/docs/tokenizers/v0.20.3/en/api/trainers) and [count/TF-IDF representations and their limitations](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction).


In [ ]:
scores.to_csv(results_folder / "cv_scores.csv", index=False)
summary.to_csv(results_folder / "summary.csv", index=False)
oof.to_csv(results_folder / "oof_predictions.csv", index=False)
token_diagnostics.to_csv(results_folder / "token_diagnostics.csv", index=False)
vocabulary_deltas.to_csv(results_folder / "vocabulary_deltas.csv", index=False)
architecture_deltas.to_csv(results_folder / "architecture_deltas.csv", index=False)
pd.DataFrame({"id": ids, "gold": y, "fold": fold_assignment}).to_csv(results_folder / "folds.csv", index=False)
(results_folder / "data_audit.json").write_text(json.dumps(audit, indent=2))
settings = {"status": run_label, "dataset_revision": dataset_revision,
    "examples_per_class": examples_per_class, "seed": seed, "folds": num_folds,
    "vocabulary_sizes": vocabulary_sizes, "architecture_control_vocab": architecture_vocab,
    "min_frequency": min_frequency, "epochs": num_epochs, "hidden_dim": hidden_dim,
    "second_hidden_dim": second_hidden_dim, "dropout": dropout, "weight_decay": weight_decay,
    "batch_size": batch_size, "learning_rate": learning_rate, "logistic_C": 1.0,
    "controls": {"width": 32, "dropout": 0.5, "weight_decay": 0.001},
    "official_test_accessed": False, "evaluation": "exploratory shared five-fold CV",
    "baseline_notebook_sha256": "c0489e407bda3d118452c4743bba82f7500ce1921b9e82fda845fe5f54acd521",
    "packages": {name: version(name) for name in
                 ["numpy", "pandas", "scipy", "scikit-learn", "torch", "tokenizers", "datasets"]}}
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print("Saved local experiment results:", results_folder.relative_to(root))